# scRNA-seq: Quality Control and Preprocessing

**Tier 3 — Applied Bioinformatics | Module 30 · Notebook 1**

*Prerequisites: Module 03 (RNA-seq Analysis), Module 07 (Machine Learning for Biology)*

---

**By the end of this notebook you will be able to:**
1. Explain the scRNA-seq experimental workflow (droplet-based 10x Genomics vs plate-based Smart-seq)
2. Generate a count matrix from raw FASTQ files using Cell Ranger or STARsolo
3. Perform quality control: filter cells by UMI count, gene count, and % mitochondrial reads
4. Normalize (library-size, scran) and log-transform count data
5. Identify highly variable genes (HVGs) for downstream analysis



**Key resources:**
- [Scanpy tutorials](https://scanpy.readthedocs.io/en/stable/tutorials.html)
- [Seurat tutorials](https://satijalab.org/seurat/articles/pbmc3k_tutorial)
- [Harvard HBC scRNA-seq training](https://hbctraining.github.io/scRNA-seq_online/)
- [Best practices for scRNA-seq (Luecken & Theis 2019)](https://www.embopress.org/doi/full/10.15252/msb.20188746)

---

[< Previous: Graph Neural Networks for Molecular Property Pr...](../29_Cheminformatics_Drug_Discovery/04_molecular_gnn.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: scRNA-seq: Dimensionality Reduction and Clustering >](02_dimensionality_reduction.ipynb)

---

## Why single-cell preprocessing matters

Bulk RNA-seq averages gene expression over thousands of cells, masking rare populations and cellular heterogeneity. A tumor biopsy with 5% cancer-stem-like cells will show those cells' signatures diluted 20-fold. scRNA-seq resolves this by profiling each cell independently — but the raw data is exceptionally noisy: dropouts (zero counts for expressed genes), ambient RNA contamination, and doublets (two cells captured as one) all corrupt the signal if not addressed. This notebook walks through every decision in the preprocessing pipeline, explaining *why* each step is necessary and what goes wrong if you skip it.

## Environment setup and synthetic data

This notebook uses synthetic count matrix data that mimics a real 10x Genomics PBMC dataset. All code runs without downloading external files.

## Common preprocessing pitfalls

- **Mitochondrial threshold too strict**: setting pct_counts_mt < 5% in metabolically active cells (cardiomyocytes, hepatocytes) will discard real cells. Always look at the joint distribution with total counts.
- **Skipping `var_names_make_unique()`**: duplicate gene symbols (e.g., two ENSG IDs mapped to the same symbol) cause cryptic downstream errors in scanpy operations.
- **Normalizing before QC filtering**: including low-quality cells biases the normalization target. Filter first, then normalize.
- **HVG selection on raw counts**: highly variable gene selection must be done on normalized (but not scaled) data. Scaling before HVG selection inflates dispersion estimates.

## Quick demo: Build a synthetic scRNA-seq AnnData object

We create a 500-cell × 2000-gene count matrix that mimics a 10x Genomics PBMC experiment. Five cell populations are embedded with distinct marker gene programs.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

n_cells = 500
n_genes = 2000
n_mt_genes = 13  # human mtDNA encodes 13 proteins

# Gene names: first 13 are mitochondrial (MT-), rest are nuclear
gene_names = [f"MT-{g}" for g in ["ND1","ND2","COX1","COX2","ATP8","ATP6","COX3",
                                    "ND3","ND4L","ND4","ND5","ND6","CYB"]]
gene_names += [f"GENE{i:04d}" for i in range(n_genes - n_mt_genes)]

# Five cell types with different baseline expression levels
cell_types = ["T_cell", "B_cell", "Monocyte", "NK_cell", "Dendritic"]
cells_per_type = n_cells // len(cell_types)
cell_type_labels = np.repeat(cell_types, cells_per_type)

# Base count matrix: negative binomial draws
counts = rng.negative_binomial(n=2, p=0.9, size=(n_cells, n_genes)).astype(np.float32)

# Add cell-type-specific marker programs
type_idx = {ct: np.where(cell_type_labels == ct)[0] for ct in cell_types}
# T cells: high CD3D (gene 50), CD3E (gene 51)
counts[np.ix_(type_idx["T_cell"], [50, 51, 52])] += rng.poisson(15, (cells_per_type, 3))
# B cells: high CD79A (gene 100), MS4A1 (gene 101)
counts[np.ix_(type_idx["B_cell"], [100, 101])] += rng.poisson(20, (cells_per_type, 2))
# Monocytes: high LYZ (gene 150), CST3 (gene 151)
counts[np.ix_(type_idx["Monocyte"], [150, 151])] += rng.poisson(25, (cells_per_type, 2))

# Inject 20 low-quality cells (high MT%, low counts)
low_q_idx = rng.choice(n_cells, 20, replace=False)
counts[low_q_idx, :n_mt_genes] += rng.poisson(50, (20, n_mt_genes))
counts[low_q_idx, n_mt_genes:] = counts[low_q_idx, n_mt_genes:] * 0.1

# Inject 5 doublets (very high total counts)
doublet_idx = rng.choice(n_cells, 5, replace=False)
counts[doublet_idx, :] *= 2.5

# Build AnnData object (the core data structure in scanpy)
obs = pd.DataFrame({'cell_type_truth': cell_type_labels}, 
                   index=[f"CELL_{i:04d}" for i in range(n_cells)])
var = pd.DataFrame({'gene_name': gene_names,
                    'is_mt': [g.startswith('MT-') for g in gene_names]},
                   index=gene_names)

adata = ad.AnnData(X=counts, obs=obs, var=var)
print(f"AnnData object: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Mitochondrial genes: {adata.var['is_mt'].sum()}")
print(f"\nData type: {type(adata.X)}")
print(f"Memory: {adata.X.nbytes / 1e6:.1f} MB (dense); real 10x data uses sparse scipy.csr_matrix)")

## 2. Generating Count Matrices

### From FASTQ to count matrix
Raw reads from a 10x experiment are processed by **Cell Ranger** (commercial) or **STARsolo** (open-source, recommended):

1. **Barcode correction**: Cell barcodes matched against a whitelist (~6 million valid 10x barcodes), allowing 1 Hamming distance
2. **Alignment**: Reads aligned to the genome using STAR with splice-aware alignment
3. **UMI counting**: Reads sharing cell barcode + gene + UMI collapsed to 1 count (UMI deduplication)
4. **Cell calling**: EmptyDrops (Cell Ranger 3+) models the ambient RNA distribution and uses a statistical test to call cells vs empty droplets

### The three output files (MEX / Market Exchange format)
- `barcodes.tsv.gz` — one cell barcode per line
- `features.tsv.gz` — Ensembl ID, gene symbol, feature type per row
- `matrix.mtx.gz` — sparse matrix (row=gene, col=cell, val=UMI count)

Loading in scanpy is one line, but `var_names_make_unique()` is essential — multiple Ensembl IDs can map to the same gene symbol (e.g., MATR3 appears twice in hg38), which causes silent errors later.

**Read depth recommendations**: 20,000–50,000 reads/cell for robust clustering; 500,000+ reads/cell for rare isoform detection. Saturation curves in Cell Ranger QC report show whether sequencing depth is sufficient.

In [ ]:
# Demonstrate what AnnData looks like after loading real 10x data
# (using our synthetic object from above)

# Key AnnData slots:
# adata.X       -- count matrix (n_cells x n_genes), usually scipy sparse CSR
# adata.obs     -- per-cell metadata DataFrame  
# adata.var     -- per-gene metadata DataFrame
# adata.obsm    -- per-cell embeddings (PCA, UMAP stored here as 'X_pca', 'X_umap')
# adata.uns     -- unstructured metadata (color palettes, clustering parameters)
# adata.layers  -- additional count matrices (e.g. 'raw_counts' before normalization)

# Save raw counts before any modification (important for downstream DE testing)
import scipy.sparse as sp
adata.layers['raw_counts'] = sp.csr_matrix(adata.X.copy())

print("AnnData structure:")
print(f"  .X shape: {adata.X.shape}")
print(f"  .obs columns: {list(adata.obs.columns)}")
print(f"  .var columns: {list(adata.var.columns)}")
print(f"\nFirst 5 cell barcodes:")
print(adata.obs_names[:5].tolist())
print(f"\nFirst 5 gene names:")
print(adata.var_names[:5].tolist())

## 3. Quality Control Metrics

Three metrics capture the most common data quality problems:

| Metric | What it measures | Low = problem | High = problem |
|--------|-----------------|---------------|----------------|
| `n_genes_by_counts` | Number of distinct genes detected | Empty droplet or dead cell | Doublet (two cells) |
| `total_counts` | Total UMI count per cell (library size) | Dead/lysed cell | Doublet |
| `pct_counts_mt` | % UMIs from mitochondrial genes | — | Dying cell (cytoplasmic RNA leaked, MT RNA retained) |

### Why high MT% indicates dying cells
When a cell ruptures, cytoplasmic mRNA leaks out of the droplet before being captured, but mitochondria (membrane-enclosed) remain intact and their transcripts are captured. A cell with 40% MT reads has almost certainly lost its cytoplasmic transcriptome — it is not a biological MT-rich cell type.

### MAD-based filtering (preferred over fixed thresholds)
The median absolute deviation (MAD) approach adapts thresholds to each dataset's distribution:
- Cells with `total_counts` < median − 3×MAD are filtered
- Cells with `pct_counts_mt` > median + 3×MAD are filtered
This is more principled than arbitrary cutoffs like "< 200 genes" or "> 20% MT".

In [ ]:
# Compute QC metrics manually to understand what scanpy's sc.pp.calculate_qc_metrics does
import numpy as np
import scipy.sparse as sp

X = adata.X if not sp.issparse(adata.X) else adata.X.toarray()
mt_mask = adata.var['is_mt'].values

adata.obs['total_counts'] = X.sum(axis=1)
adata.obs['n_genes_by_counts'] = (X > 0).sum(axis=1)
mt_counts = X[:, mt_mask].sum(axis=1)
adata.obs['pct_counts_mt'] = (mt_counts / adata.obs['total_counts'] * 100)

print("QC metric summary:")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe().round(2))

# MAD-based thresholds (robust, adapts to each dataset)
def mad_outlier(series, n_mad=3, direction="high"):
    median = series.median()
    mad = (series - median).abs().median()
    if direction == "high":
        return series > median + n_mad * mad
    elif direction == "low":
        return series < median - n_mad * mad

fail_total = mad_outlier(adata.obs["total_counts"], direction="low")
fail_genes = mad_outlier(adata.obs["n_genes_by_counts"], direction="low")
fail_mt    = mad_outlier(adata.obs["pct_counts_mt"], direction="high")

quality_mask = ~(fail_total | fail_genes | fail_mt)
print(f"\nCells before QC: {adata.n_obs}")
print(f"  Failed total_counts (low): {fail_total.sum()}")
print(f"  Failed n_genes (low):      {fail_genes.sum()}")
print(f"  Failed MT% (high):         {fail_mt.sum()}")
print(f"Cells after QC: {quality_mask.sum()}")

# Joint scatter plot: key diagnostic
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ["red" if not q else "steelblue" for q in quality_mask]
axes[0].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
                c=colors, alpha=0.4, s=8)
axes[0].set_xlabel("Total UMI counts")
axes[0].set_ylabel("Genes detected")
axes[0].set_title("QC: genes vs counts (red = filtered)")

axes[1].scatter(adata.obs["total_counts"], adata.obs["pct_counts_mt"],
                c=colors, alpha=0.4, s=8)
axes[1].set_xlabel("Total UMI counts")
axes[1].set_ylabel("% Mitochondrial counts")
axes[1].set_title("QC: MT% vs counts")

plt.tight_layout()
plt.savefig("qc_scatter.png", dpi=100, bbox_inches="tight")
plt.show()

# Apply filter (subset AnnData)
adata = adata[quality_mask].copy()
print(f"\nFiltered to {adata.n_obs} cells x {adata.n_vars} genes")


## 4. Normalization and Log-Transformation

### Why normalize?
Each cell has a different total UMI count (library size) due to technical variation — more cDNA molecules were captured from some cells than others. Without normalization, a gene appearing at 100 counts/cell in a 10,000-count cell has the same normalized expression as a gene at 200 counts in a 20,000-count cell.

### Library-size normalization (CPM-style)
Divide each cell's counts by its total, then multiply by a scale factor (10,000 by convention, making units "counts per 10k" or CP10K):

```
normalized_count = raw_count / cell_total * 10,000
```

This is `sc.pp.normalize_total(adata, target_sum=1e4)` in scanpy.

### Log1p transformation
After normalization, counts span several orders of magnitude (0.001 to 1000). Log1p compression `log(x+1)`:
- Brings the distribution closer to normal (required for PCA)
- The +1 prevents log(0) for zero counts
- Makes fold-changes between groups symmetric: doubling and halving are equal distances

### scran pooling-based normalization (alternative)
scran (Lun et al. 2016) pools cells to estimate size factors, then normalizes each cell by its estimated factor. More statistically rigorous than library-size normalization, particularly for datasets with strong composition bias (e.g. one cell type dominates). Implemented in `sc.external.pp.scran_normalization()`.

### SCTransform / analytic Pearson residuals (Seurat approach)
Fits a regularized negative binomial regression model to remove the mean-variance trend inherent to count data. Equivalent to variance-stabilizing normalization. For scanpy users: `sc.experimental.pp.normalize_pearson_residuals()`.

In [ ]:
# Normalize and log-transform
import numpy as np

# Step 1: Library-size normalization
# Each cell is scaled so total counts = 10,000
cell_totals = adata.obs["total_counts"].values.reshape(-1, 1)
X_norm = adata.X / cell_totals * 1e4

# Step 2: Log1p transformation
X_log = np.log1p(X_norm)

# Store in adata (this overwrites .X)
adata.X = X_log
print(f"After normalization + log1p:")
print(f"  Min value: {adata.X.min():.3f}")
print(f"  Max value: {adata.X.max():.3f}")
print(f"  Mean (all cells): {adata.X.mean():.3f}")

# Verify: a gene with 500 raw UMIs in a 10,000-count cell
raw = 500
total = 10000
normalized = np.log1p(raw / total * 1e4)
print(f"\nExample: 500 raw UMIs in a 10k-count cell -> log1p(500/10000*10000) = log1p(500) = {normalized:.3f}")

# Compare raw count distribution vs log-normalized
import matplotlib.pyplot as plt
gene_idx = 150  # LYZ-like marker gene

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(adata.layers["raw_counts"].toarray()[:, gene_idx].flatten(), 
             bins=40, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Raw UMI counts")
axes[0].set_title(f"Gene {adata.var_names[gene_idx]}: raw counts")
axes[0].set_ylabel("Cells")

axes[1].hist(adata.X[:, gene_idx].flatten() if not hasattr(adata.X, "toarray") 
             else adata.X.toarray()[:, gene_idx].flatten(),
             bins=40, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Log-normalized expression")
axes[1].set_title(f"Gene {adata.var_names[gene_idx]}: log-normalized")

plt.tight_layout()
plt.savefig("normalization.png", dpi=100, bbox_inches="tight")
plt.show()
print("Raw counts are right-skewed; log-normalized values are closer to normal")

## 5. Highly Variable Gene Selection

### The mean-variance relationship problem
In scRNA-seq, high-count genes are inherently more variable simply because there are more counts to vary. If we ran PCA on all 20,000+ genes, the principal components would be dominated by highly expressed but biologically uninformative genes (ribosomal proteins, housekeeping genes). We need to select genes that vary *more than expected* given their mean expression level.

### How HVG selection works (Seurat flavor)
1. For each gene, compute mean expression and dispersion (variance/mean, i.e. coefficient of variation)
2. Bin genes into 20 groups by mean expression
3. Within each bin, normalize dispersion by the bin's mean and standard deviation
4. Select genes in the top tail of normalized dispersion (typically 2,000–5,000 genes)

The Seurat method is the default in scanpy (`flavor="seurat"`). The Cell Ranger flavor uses a slightly different binning strategy and is appropriate when you have Cell Ranger count matrices.

### How many HVGs to select?
- **2,000–3,000**: appropriate for most datasets; captures major cell type differences
- **5,000–10,000**: for subtle sub-populations; increases compute time and noise
- **Too few (< 500)**: misses rare cell type markers; over-regularizes

After selecting HVGs, we subset `adata` to only these genes before PCA. The full matrix is retained in `adata.raw` for downstream DE analysis.

In [ ]:
# Highly Variable Gene (HVG) selection
# Manual implementation of the Seurat flavor to understand the algorithm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X = adata.X if not hasattr(adata.X, "toarray") else adata.X.toarray()

# Compute per-gene statistics
mean_expr = X.mean(axis=0)
var_expr = X.var(axis=0)
# Dispersion = variance / mean (coefficient of variation-like, on log-normalized data)
# Use log of dispersion to stabilize
with np.errstate(divide="ignore", invalid="ignore"):
    dispersion = np.where(mean_expr > 0, var_expr / (mean_expr + 1e-10), 0)

# Bin genes by mean expression (20 bins)
n_bins = 20
gene_df = pd.DataFrame({
    "gene": adata.var_names,
    "mean": mean_expr,
    "dispersion": dispersion,
    "log_mean": np.log1p(mean_expr),
})

gene_df["bin"] = pd.cut(gene_df["log_mean"], bins=n_bins, labels=False)

# Within each bin, normalize dispersion by bin mean/std
gene_df["dispersion_norm"] = gene_df.groupby("bin")["dispersion"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-10)
)

# Select top 1500 HVGs by normalized dispersion (excluding MT genes)
non_mt = ~adata.var["is_mt"].values
gene_df["is_mt"] = adata.var["is_mt"].values

hvg_candidates = gene_df[~gene_df["is_mt"]].nlargest(1500, "dispersion_norm")
adata.var["highly_variable"] = adata.var_names.isin(hvg_candidates["gene"])
adata.var["mean_counts"] = mean_expr
adata.var["dispersions_norm"] = gene_df["dispersion_norm"].values

print(f"Selected {adata.var['highly_variable'].sum()} highly variable genes")
print(f"(excluded {adata.var['is_mt'].sum()} mitochondrial genes from HVG selection)")

# Plot: mean expression vs normalized dispersion (the key HVG diagnostic)
fig, ax = plt.subplots(figsize=(8, 5))
non_hvg_mask = ~adata.var["highly_variable"].values
hvg_mask = adata.var["highly_variable"].values

ax.scatter(gene_df.loc[non_hvg_mask, "log_mean"], 
           gene_df.loc[non_hvg_mask, "dispersion_norm"],
           s=4, alpha=0.3, color="lightgray", label="Other genes")
ax.scatter(gene_df.loc[hvg_mask, "log_mean"],
           gene_df.loc[hvg_mask, "dispersion_norm"],
           s=6, alpha=0.7, color="tomato", label=f"HVGs (n={hvg_mask.sum()})")
ax.set_xlabel("Mean log-normalized expression")
ax.set_ylabel("Normalized dispersion")
ax.set_title("Highly Variable Gene Selection")
ax.legend()
plt.tight_layout()
plt.savefig("hvg_selection.png", dpi=100, bbox_inches="tight")
plt.show()

# Subset to HVGs for downstream analysis
# But FIRST save the full matrix in adata.raw for differential expression
# In real scanpy workflow: adata.raw = adata before subsetting
print(f"\nSubsetting to {adata.var['highly_variable'].sum()} HVGs...")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
print(f"Shape after HVG selection: {adata_hvg.shape}")

## Summary: The preprocessing pipeline

The standard scanpy scRNA-seq preprocessing pipeline follows this exact order:

```
Raw 10x data (MEX format)
    ↓
sc.read_10x_mtx() + var_names_make_unique()
    ↓
Save raw counts: adata.layers["raw_counts"] = adata.X.copy()
    ↓  
QC metrics: n_genes, total_counts, pct_counts_mt
    ↓
Filter low-quality cells (MAD-based or manual thresholds)
    ↓
Normalize: sc.pp.normalize_total(target_sum=1e4) 
    ↓
Log-transform: sc.pp.log1p()
    ↓
HVG selection: sc.pp.highly_variable_genes(n_top_genes=2000)
    ↓
Save to disk: adata.write_h5ad("preprocessed.h5ad")
    ↓
Next: PCA → Neighbors → UMAP → Clustering (Notebook 2)
```

**Decisions made in this notebook and their downstream effects:**
- MT threshold: too strict → lose metabolically active cells; too loose → include dying cells in every cluster
- Target sum: 10,000 is convention; some analyses use 1e6 (CPM) — be consistent within a project
- n_top_genes: fewer = faster but may miss rare marker genes
- Save raw counts in `.layers["raw_counts"]` — required for proper differential expression testing (DESeq2, edgeR expect raw integer counts, not log-normalized values)

In [ ]:
# Save processed AnnData object for Notebook 2
# h5ad format (HDF5-based) is efficient for sparse matrices and preserves all metadata

adata_hvg.write_h5ad("/tmp/scrna_preprocessed.h5ad")
print(f"Saved to /tmp/scrna_preprocessed.h5ad")
print(f"Final shape: {adata_hvg.n_obs} cells x {adata_hvg.n_vars} HVGs")
print(f"\nStored layers: {list(adata_hvg.layers.keys())}")
print(f"Obs metadata: {list(adata_hvg.obs.columns)}")
print(f"Var metadata: {list(adata_hvg.var.columns)}")

# Verify we can reload it
import anndata as ad
check = ad.read_h5ad("/tmp/scrna_preprocessed.h5ad")
print(f"\nReload check: {check}")

---

[< Previous: Graph Neural Networks for Molecular Property Pr...](../29_Cheminformatics_Drug_Discovery/04_molecular_gnn.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: scRNA-seq: Dimensionality Reduction and Clustering >](02_dimensionality_reduction.ipynb)

---